# 01: RAG Retrieval Analysis

This notebook explores the local golden evaluation dataset, illustrates retrieval trade-offs across dense, sparse, and hybrid modes, and inspects chunk sizing behavior using the persisted vector-store metadata when available.

## Section 1 — Setup & Imports

This cell imports the plotting and analysis stack, points Python at the backend package, and defines the local paths used by the notebook.

In [ ]:
import sys, json, time, os
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'backend'))
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import pandas as pd
from IPython.display import display
sns.set_theme(style='darkgrid', palette='muted')

DATASET_PATH = Path('..') / 'backend' / 'data' / 'golden_evaluation_dataset.jsonl'
CHUNKS_PATH = Path('..') / 'backend' / 'data' / 'vector_store' / 'chunks.pkl'
CHUNK_SIZE = 512
np.random.seed(42)

print(f'Dataset path: {DATASET_PATH.resolve()}')
print(f'Chunk metadata path: {CHUNKS_PATH.resolve()}')


: 

## Section 2 — Load Golden Dataset

This cell loads the five clinical Q&A pairs from the golden dataset and presents them as a styled pandas table for quick inspection.

In [ ]:
records = []
with DATASET_PATH.open('r', encoding='utf-8') as handle:
    for line in handle:
        if not line.strip():
            continue
        row = json.loads(line)
        context_text = ' '.join(row.get('context') or row.get('contexts') or [])
        preview = context_text[:120] + ('...' if len(context_text) > 120 else '')
        records.append({
            'Question': row['question'],
            'Ground Truth': row.get('ground_truth', ''),
            'Context Preview': preview,
        })

questions_df = pd.DataFrame(records)
questions_df.style.hide(axis='index').set_properties(**{'text-align': 'left'}).set_caption('Golden clinical Q&A dataset (n=5)')


## Section 3 — Retrieval Mode Comparison

This cell creates an illustrative benchmark table for the four retrieval modes. The values are deliberately labeled as illustrative so they are not confused with the measured benchmark output from `backend/scripts/benchmark_retrieval.py`.

In [ ]:
data = {
    'Mode': ['FAISS Only', 'BM25 Only', 'Hybrid (RRF)', 'Hybrid + Reranker'],
    'Precision@5': [0.72, 0.65, 0.84, 0.89],
    'MRR':         [0.68, 0.61, 0.79, 0.85],
    'Latency_ms':  [45,   12,   58,   210],
}
comparison_df = pd.DataFrame(data)
illustrative_note = 'Illustrative results — run benchmark_retrieval.py for real numbers'
print(illustrative_note)
display(comparison_df)

quality_df = comparison_df.melt(
    id_vars='Mode',
    value_vars=['Precision@5', 'MRR'],
    var_name='Metric',
    value_name='Score',
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=quality_df, x='Mode', y='Score', hue='Metric', ax=ax)
ax.set_ylim(0, 1.0)
ax.set_title('Illustrative Retrieval Quality by Mode')
ax.set_xlabel('Retrieval mode')
ax.set_ylabel('Score')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=comparison_df, x='Latency_ms', y='Mode', orient='h', ax=ax, color=sns.color_palette('muted')[2])
ax.set_title('Illustrative Retrieval Latency by Mode')
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Retrieval mode')
plt.tight_layout()
plt.show()

comparison_df


## Section 4 — Chunk Analysis

This cell inspects the persisted FAISS chunk metadata from `chunks.pkl`. If the file is missing, empty, or unreadable, it falls back to synthetic chunk-length values centered around the configured 512-word chunk target.

In [ ]:
import pickle

chunk_lengths = []
data_source = 'real chunk metadata'

try:
    if CHUNKS_PATH.exists() and CHUNKS_PATH.stat().st_size > 0:
        with CHUNKS_PATH.open('rb') as handle:
            chunk_payload = pickle.load(handle)
        if isinstance(chunk_payload, list):
            for chunk in chunk_payload:
                text = chunk.get('chunk_text') or chunk.get('text') or ''
                if str(text).strip():
                    chunk_lengths.append(len(str(text).split()))
except Exception as exc:
    print(f'Falling back to synthetic chunk lengths: {exc}')

if not chunk_lengths:
    data_source = 'synthetic fallback'
    chunk_lengths = np.clip(np.random.normal(loc=450, scale=80, size=20).round().astype(int), 50, None)

chunk_lengths_df = pd.DataFrame({'chunk_length_words': chunk_lengths})

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(
    chunk_lengths_df['chunk_length_words'],
    bins=min(12, max(5, len(chunk_lengths_df))),
    kde=False,
    ax=ax,
    color=sns.color_palette('muted')[0],
)
ax.axvline(CHUNK_SIZE, color='crimson', linestyle='--', linewidth=2)
ax.annotate(
    'Target chunk size: 512 words',
    xy=(CHUNK_SIZE, ax.get_ylim()[1] * 0.75),
    xytext=(CHUNK_SIZE + 20, ax.get_ylim()[1] * 0.85),
    arrowprops={'arrowstyle': '->', 'color': 'crimson'},
    color='crimson',
)
ax.set_title(f'Chunk Length Distribution ({data_source})')
ax.set_xlabel('Chunk length (words)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

chunk_lengths_df.describe()


## Section 5 — Trade-off Analysis

This cell visualizes the quality-versus-latency trade-off directly. Point size is scaled by MRR, and the upper-left region is annotated as the desirable high-quality, low-latency operating zone.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=comparison_df,
    x='Latency_ms',
    y='Precision@5',
    size='MRR',
    sizes=(200, 650),
    hue='Mode',
    palette='muted',
    legend=False,
    ax=ax,
)

for _, row in comparison_df.iterrows():
    ax.text(row['Latency_ms'] + 3, row['Precision@5'] + 0.005, row['Mode'], fontsize=10)

ax.axvline(60, linestyle='--', color='gray', alpha=0.7)
ax.axhline(0.80, linestyle='--', color='gray', alpha=0.7)
ax.text(
    18,
    0.865,
    'Sweet Spot',
    fontsize=12,
    fontweight='bold',
    color='darkgreen',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='darkgreen', alpha=0.8),
)
ax.set_title('Retrieval Trade-off: Latency vs Precision')
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Precision@5')
plt.tight_layout()
plt.show()

comparison_df[['Mode', 'Latency_ms', 'Precision@5', 'MRR']]


## Section 6 — Key Takeaways

- Hybrid + Reranker achieves the highest quality in this illustrative comparison, reaching 0.89 Precision@5, which is about 24% higher than the 0.72 FAISS-only baseline.
- BM25 is the fastest mode and remains useful for keyword-heavy clinical queries where literal string matching matters.
- RRF fusion adds only about 13 ms over FAISS-only latency in the illustrative comparison while improving Precision@5 from 0.72 to 0.84.
- The reranker adds about 152 ms over Hybrid (RRF), but that extra latency can be justified when clinical safety and ranking quality matter more than raw speed.